# HiProbCBM - PROBCBM - A3

**Kondisi:** CUB-200-2011 / ResNet18  
**Pembanding publikasi:** ProbCBM  

Notebook ini dapat dijalankan mandiri setelah dataset pada Drive sudah siap.

Jalankan sel dari atas ke bawah. Training selalu memakai seed 42, 43, 44 secara berurutan dan checkpoint disimpan di Google Drive. Jika runtime terputus, siapkan ulang notebook ini lalu jalankan kembali sel training; `--resume auto` melanjutkan checkpoint yang valid.


## 1. Mount Drive dan tetapkan folder tesis

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import torch

DRIVE_ROOT = Path("/content/drive/MyDrive/TESIS AZHIM")
assert DRIVE_ROOT.is_dir(), f"Folder tesis tidak ditemukan: {DRIVE_ROOT}"
print("Drive siap:", DRIVE_ROOT)


## 2. Clone/perbarui repo dan instal dependensi

In [ ]:
%cd /content
!test -d HiProbCBM || git clone https://github.com/Azhimanich/HiProbCBM.git
%cd /content/HiProbCBM
!git pull --ff-only
!pip install -e . -q


## Siapkan CUB dari Drive

Sel ini hanya memeriksa metadata yang sudah ada di Drive dan membuat symlink ke runtime. File gambar tidak disalin.


In [ ]:
import os
import pickle

DRIVE_CUB_ROOT = DRIVE_ROOT / "CUB_200_2011"
assert (DRIVE_CUB_ROOT / "images").is_dir(), f"Folder CUB tidak ditemukan: {DRIVE_CUB_ROOT}"
metadata = DRIVE_CUB_ROOT / "metadata"
needed = [metadata / name for name in ("train.pkl", "val.pkl", "test.pkl")]
assert all(path.is_file() for path in needed), "metadata/train.pkl, val.pkl, test.pkl harus ada di Drive."

# Normalisasi ini idempotent: hanya menulis ulang pkl bila masih ada prefix lama.
marker = "CUB_200_2011/images/"
for path in needed:
    with path.open("rb") as handle:
        records = pickle.load(handle)
    changed = sum(marker in record["img_path"] for record in records)
    if changed:
        for record in records:
            if marker in record["img_path"]:
                record["img_path"] = record["img_path"].split(marker, 1)[1]
        with path.open("wb") as handle:
            pickle.dump(records, handle)
        print(f"Dinormalisasi: {path.name} ({changed} path)")
    else:
        print(f"OK: {path.name} ({len(records)} record)")

WORK_ROOT = Path("/content/HiProbCBM/datasets/CUB_200_2011")
WORK_ROOT.parent.mkdir(parents=True, exist_ok=True)
if WORK_ROOT.exists() or WORK_ROOT.is_symlink():
    assert WORK_ROOT.is_symlink() and WORK_ROOT.resolve() == DRIVE_CUB_ROOT.resolve(), (
        f"Symlink CUB tidak sesuai: {WORK_ROOT}. Jangan hapus folder non-symlink secara otomatis."
    )
else:
    os.symlink(DRIVE_CUB_ROOT, WORK_ROOT, target_is_directory=True)
print("CUB siap:", WORK_ROOT, "->", DRIVE_CUB_ROOT)


## 3. Sambungkan log/checkpoint ke Drive dan cek GPU

In [ ]:
TRAIN_LOG_DRIVE = DRIVE_ROOT / "HiProbCBM_results" / "train_log"
TRAIN_LOG_DRIVE.mkdir(parents=True, exist_ok=True)
TRAIN_LOG_LOCAL = Path("/content/HiProbCBM/train_log")
if TRAIN_LOG_LOCAL.exists() or TRAIN_LOG_LOCAL.is_symlink():
    assert TRAIN_LOG_LOCAL.is_symlink() and TRAIN_LOG_LOCAL.resolve() == TRAIN_LOG_DRIVE.resolve(), (
        f"train_log lokal bukan symlink yang diharapkan: {TRAIN_LOG_LOCAL}"
    )
else:
    os.symlink(TRAIN_LOG_DRIVE, TRAIN_LOG_LOCAL, target_is_directory=True)

def assert_persistent_training_storage():
    assert Path("/content/drive/MyDrive").is_dir(), "Google Drive belum mounted."
    assert TRAIN_LOG_LOCAL.is_symlink(), "train_log harus symlink ke Drive."
    assert TRAIN_LOG_LOCAL.resolve() == TRAIN_LOG_DRIVE.resolve(), "Target train_log berubah."

assert_persistent_training_storage()
assert torch.cuda.is_available(), "GPU belum aktif. Pilih runtime GPU sebelum training."
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint Drive:", TRAIN_LOG_DRIVE)


## 4. Jalankan training tiga seed

Sel ini menjalankan seluruh training skenario ini. Jangan menjalankan notebook training lain secara bersamaan.


In [ ]:
import subprocess
from hiprobcbm.config import Config, load_config, prepare_log_dir

PROTOKOL = "probcbm"
VARIAN = "a3"
STAGE1_CONFIG = "configs/cub_stage1_probcbm_reported_batchtopk.yaml"
STAGE2_CONFIG = "configs/cub_stage2_probcbm_reported_batchtopk.yaml"
SEEDS = [42, 43, 44]

# A1/A2 selalu mengambil hierarchy model utama; A3 memakai hierarchy BatchTopK-nya sendiri.
def stage1_root(seed):
    return f"train_log/{PROTOKOL}/a3/seed_{seed}"

def stage2_root(seed):
    return f"train_log/{PROTOKOL}/{VARIAN}/seed_{seed}"

def stage1_dir(seed):
    return prepare_log_dir(load_config(STAGE1_CONFIG), stage1_root(seed))

def stage2_dir(seed):
    values = load_config(STAGE2_CONFIG).to_dict()
    if VARIAN in {"a1", "a2"}:
        values["experiment_name"] = f"{values['experiment_name']}_ablation_{VARIAN}"
    return prepare_log_dir(Config(values), stage2_root(seed))

def run_visible(command, log_dir):
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    print("CMD:", " ".join(command))
    with (log_dir / "console.log").open("a", encoding="utf-8") as logfile:
        logfile.write("\nCMD: " + " ".join(command) + "\n")
        process = subprocess.Popen(command, cwd="/content/HiProbCBM", stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            logfile.write(line)
            logfile.flush()
        exit_code = process.wait()
    if exit_code:
        raise RuntimeError(f"Trainer berhenti (exit={exit_code}). Baca error di atas atau {log_dir / 'console.log'}.")

def stage2_command(seed):
    common = ["--config", STAGE2_CONFIG, "--stage1-log-dir", str(stage1_dir(seed)),
              "--seed", str(seed), "--log-dir", stage2_root(seed), "--resume", "auto"]
    if VARIAN in {"a1", "a2"}:
        return ["python", "scripts/run_ablation.py", *common]
    return ["python", "scripts/run_stage2.py", *common]

assert torch.cuda.is_available(), "GPU Colab belum aktif. Pilih runtime GPU sebelum menjalankan training."
assert_persistent_training_storage()
print("GPU:", torch.cuda.get_device_name(0))
print("Skenario:", PROTOKOL, "/", VARIAN, "| seeds:", SEEDS)

for seed in SEEDS:
    parent_dir = stage1_dir(seed)
    if True:
        print(f"\n{'=' * 72}\nSEED {seed}: Tahap 1 ({VARIAN})\n{'=' * 72}")
        run_visible(["python", "scripts/run_stage1.py", "--config", STAGE1_CONFIG,
                     "--seed", str(seed), "--log-dir", stage1_root(seed), "--resume", "auto"], parent_dir)
    else:
        assert (parent_dir / "pseudo_hierarchy.pt").exists(), (
            f"Hierarchy utama belum ada untuk seed {seed}: {parent_dir / 'pseudo_hierarchy.pt'}. "
            "Selesaikan notebook utama untuk protokol ini terlebih dahulu."
        )
        print(f"SEED {seed}: memakai hierarchy utama {parent_dir / 'pseudo_hierarchy.pt'}")
    assert (parent_dir / "pseudo_hierarchy.pt").exists(), f"Tahap 1 seed {seed} belum menghasilkan hierarchy."
    print(f"\n{'=' * 72}\nSEED {seed}: Tahap 2 ({VARIAN})\n{'=' * 72}")
    run_visible(stage2_command(seed), stage2_dir(seed))

print("Training tiga seed selesai. Jalankan sel evaluasi berikutnya.")


## 5. Evaluasi checkpoint terbaik dan simpan CSV

In [ ]:
import csv
import json
import statistics
from hiprobcbm.config import Config, load_config, prepare_log_dir, resolve_device
from hiprobcbm.engine.evaluate import evaluate_hiprobcbm_checkpoint
from hiprobcbm.utils.seed import set_random_seed

PROTOKOL = "probcbm"
VARIAN = "a3"
STAGE1_CONFIG = "configs/cub_stage1_probcbm_reported_batchtopk.yaml"
STAGE2_CONFIG = "configs/cub_stage2_probcbm_reported_batchtopk.yaml"
SEEDS = [42, 43, 44]

def stage1_dir(seed):
    return prepare_log_dir(load_config(STAGE1_CONFIG), f"train_log/{PROTOKOL}/a3/seed_{seed}")

def stage2_dir(seed):
    values = load_config(STAGE2_CONFIG).to_dict()
    if VARIAN in {"a1", "a2"}:
        values["experiment_name"] = f"{values['experiment_name']}_ablation_{VARIAN}"
    return prepare_log_dir(Config(values), f"train_log/{PROTOKOL}/{VARIAN}/seed_{seed}")

assert_persistent_training_storage()
device = resolve_device(0)
rows = []
for seed in SEEDS:
    parent_dir, child_dir = stage1_dir(seed), stage2_dir(seed)
    best = child_dir / "stage2_best.pth"
    manifest = child_dir / "run_manifest.json"
    hierarchy_path = parent_dir / "pseudo_hierarchy.pt"
    if not (best.exists() and manifest.exists() and hierarchy_path.exists()):
        print(f"LEWATI seed {seed}: checkpoint atau hierarchy belum lengkap.")
        continue
    cfg = Config(json.loads(manifest.read_text(encoding="utf-8"))["config"])
    set_random_seed(cfg.get("seed", seed))
    hierarchy = torch.load(hierarchy_path, map_location="cpu", weights_only=True)
    metrics = evaluate_hiprobcbm_checkpoint(cfg, best, hierarchy["subconcepts_per_concept"], device)
    rows.append({"protokol": PROTOKOL, "varian": VARIAN, "seed": seed, "checkpoint": str(best), **metrics})
    print(f"OK seed {seed}: {metrics}")

out_csv = Path("/content/HiProbCBM") / "train_log" / PROTOKOL / VARIAN / "ringkasan_tiga_seed.csv"
if rows:
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    fields = list(dict.fromkeys(key for row in rows for key in row))
    with out_csv.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    print("CSV:", out_csv)
    for metric in sorted({key for row in rows for key, value in row.items() if isinstance(value, (int, float)) and key != "seed"}):
        values = [row[metric] for row in rows if isinstance(row.get(metric), (int, float))]
        mean = statistics.mean(values)
        std = statistics.stdev(values) if len(values) > 1 else 0.0
        print(f"{metric}: {mean:.6f} +/- {std:.6f} (n={len(values)})")
else:
    print("Belum ada tiga checkpoint yang siap dievaluasi.")
